In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import os.path as osp

DS_PATH = "/kaggle/input/ethz-cil-monocular-depth-estimation-2025"

print(os.listdir(DS_PATH))

TRAIN_PATH = osp.join(DS_PATH, "train/train")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

['train_list.txt', 'test_list.txt', 'create_prediction_csv.py', 'test', 'train']


In [2]:

import subprocess
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("git_access_token")

repo_link = "https://" + secret_value_0 + "@github.com/aryansood/CIL.git"

os.chdir("/kaggle/working")

subprocess.run(["rm -rf CIL"], shell=True)
subprocess.run(["git clone -b auto_run " + repo_link], shell=True)

del repo_link

print("repo cloned")
os.chdir("/kaggle/working/CIL")

Cloning into 'CIL'...


repo cloned


In [3]:
import wandb
wandb_key = user_secrets.get_secret("wandb_api_key")

wandb.login(
    key=wandb_key
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


wandb: Currently logged in as: 10847706 (10847706-ethz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
!pip install lightning 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/818.9 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.9/818.9 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/363.4 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/363.4 MB 130.0 MB/s eta 0:00:03

   ━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.8/363.4 MB 191.8 MB/s eta 0:00:02

   ━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/363.4 MB 211.9 MB/s eta 0:00:02

...

    Found existing installation: nvidia-cudnn-cu12 9.3.0.75
    Uninstalling nvidia-cudnn-cu12-9.3.0.75:


      Successfully uninstalled nvidia-cudnn-cu12-9.3.0.75


  Attempting uninstall: nvidia-cusolver-cu12
    Found existing installation: nvidia-cusolver-cu12 11.7.3.90
    Uninstalling nvidia-cusolver-cu12-11.7.3.90:


      Successfully uninstalled nvidia-cusolver-cu12-11.7.3.90


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcugraph-cu12 24.12.0 requires pylibraft-cu12==24.12.*, but you have pylibraft-cu12 25.2.0 which is incompatible.
pylibcugraph-cu12 24.12.0 requires rmm-cu12==24.12.*, but you have rmm-cu12 25.2.0 which is incompatible.


In [5]:
from training import begin_training_loop
from models.segformer_depth import SegFormerDepthEstimator
import albumentations as A
from pathlib import Path
from datetime import timedelta


wandb.finish()
normalize_noise_augmentation = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

augmentations = [
    A.Compose([
        normalize_noise_augmentation,
        A.ToTensorV2()
    ])
]

model = SegFormerDepthEstimator(1e-4, pretrained_weights="nvidia/segformer-b4-finetuned-ade-512-512")
model.encoder = model.encoder.to("cuda")
model.decoder = model.decoder.to("cuda")

begin_training_loop(
    model = model,
    data_dir = Path(TRAIN_PATH),
    random_split=False,
    train_split=pd.read_csv("train_split.csv")["file_name"].to_list(),
    val_split=pd.read_csv("val_split.csv")["file_name"].to_list(),
    augmentations = augmentations,
    batch_size=4,
    num_worker=2,
    max_training_duration = timedelta(hours=8),
    check_point_every_step = 3000,
)

wandb.finish()


/usr/local/lib/python3.11/dist-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.5' (you have '2.0.4'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


2025-04-22 17:05:54.234624: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745341554.429271      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745341554.482233      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


config.json:   0%|          | 0.00/6.89k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/257M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/257M [00:00<?, ?B/s]

...

/usr/local/lib/python3.11/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 4. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 2. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Validation: |          | 0/? [00:00<?, ?it/s]